In [1]:
# Check if the file is actually there
import os

print("Listing /app:")
for name in sorted(os.listdir("/app")):
    print(" -", name)

Listing /app:
 - .dockerignore
 - Dockerfile
 - README.md
 - checkpoints
 - data
 - notebooks
 - outputs
 - requirements.txt


In [2]:
# ======================================================
# FULL BAGGING ENSEMBLE (FINAL PRODUCTION VERSION)
# ======================================================

import os
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.classification import RandomForestClassifier
from py4j.protocol import Py4JError

# 1. INITIALIZE SPARK SESSION
# ------------------------------------------------------
# Stop any existing SparkSession to avoid conflicts
if 'spark' in locals() and spark:
    try:
        spark.stop()
        print("Stopped existing Spark Session.")
    except Py4JError as e:
        print(f"Could not stop existing Spark session gracefully: {e}")
        print("       Proceeding to create a new one, but previous session might be in an inconsistent state.")

spark = (
    SparkSession.builder
    .appName("CSCI316-Bagging-Final")
    .master("local[*]")
    .config("spark.driver.memory", "10g") # Increased memory for full dataset
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created Successfully!")

# 2. DATA LOADING (Assuming already unzipped)
# ------------------------------------------------------
BASE_DIR = "/app"
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")

DATA_PATH = os.path.join(DATA_PROCESSED, "valuation_ml_cv_ready.parquet")

if os.path.exists(DATA_PATH):
    df = spark.read.parquet(DATA_PATH)
    df = df.cache() # Cache is essential for 10-fold CV
    print(f"Full Dataset Loaded. Total Rows: {df.count()}")
else:
    raise FileNotFoundError(
        f"Could not find Parquet folder at: {DATA_PATH}\n"
        "Please run 03_cross_validation.ipynb first to generate valuation_ml_cv_ready.parquet."
    )

# 3. DETECT COLUMNS
# ------------------------------------------------------
cols = set(df.columns)
LABEL_COL = "label" if "label" in cols else "target"
FEATURES_COL = "features" if "features" in cols else "feature_vector"
FOLD_COL = "fold_id" if "fold_id" in cols else "fold"

# 4. DEFINE BAGGING & METRICS
# ------------------------------------------------------
def bagging_train_predict(train_df, test_df, features_col, label_col,
                          num_trees=50, max_depth=10, seed=42):
    # Standard Bagging uses deeper trees and more of them
    rf = RandomForestClassifier(
        featuresCol=features_col, labelCol=label_col, predictionCol="prediction",
        numTrees=num_trees, maxDepth=max_depth, featureSubsetStrategy="all", seed=seed
    )
    model = rf.fit(train_df)
    return model.transform(test_df).select(label_col, "prediction")

def compute_metrics(pred_df, label_col, pred_col):
    total = pred_df.count()
    correct = pred_df.filter(F.col(label_col) == F.col(pred_col)).count()
    accuracy = correct / total if total > 0 else 0.0

    tp = pred_df.filter(F.col(label_col) == F.col(pred_col)).groupBy(label_col).count().withColumnRenamed("count", "tp")
    pred_counts = pred_df.groupBy(pred_col).count().withColumnRenamed(pred_col, "cls").withColumnRenamed("count", "pred_total")
    actual_counts = pred_df.groupBy(label_col).count().withColumnRenamed(label_col, "cls").withColumnRenamed("count", "actual_total")

    per = actual_counts.join(pred_counts, "cls", "full").join(tp.withColumnRenamed(label_col, "cls"), "cls", "left").na.fill(0)
    per = per.withColumn("precision", F.when(F.col("pred_total") == 0, 0.0).otherwise(F.col("tp") / F.col("pred_total"))) \
             .withColumn("recall", F.when(F.col("actual_total") == 0, 0.0).otherwise(F.col("tp") / F.col("actual_total")))
    per = per.withColumn("f1", F.when((F.col("precision") + F.col("recall")) == 0, 0.0) \
                     .otherwise(2 * F.col("precision") * F.col("recall") / (F.col("precision") + F.col("recall"))))


    rows = per.collect()
    return {"accuracy": accuracy, "precision": sum(r["precision"] for r in rows) / len(rows),
            "recall": sum(r["recall"] for r in rows) / len(rows), "f1": sum(r["f1"] for r in rows) / len(rows)}

# 5. FULL 10-FOLD CROSS-VALIDATION
# ------------------------------------------------------
NUM_TREES = 50          # Standard for stable bagging
MAX_DEPTH = 10          # Deeper trees for better learning
# No SAMPLE_FRACTION here - we are using 100% of the data

all_folds = [r[FOLD_COL] for r in df.select(FOLD_COL).distinct().orderBy(FOLD_COL).collect()]

print(f"Running FULL Bagging Ensemble on all {len(all_folds)} folds...")
print(f"Using 100% of Training Data per fold.")
print(f"Forest: {NUM_TREES} trees, Depth {MAX_DEPTH}")
print("=" * 60)

results = []
start_time = time.time()

for i, k in enumerate(all_folds, start=1):
    fold_start = time.time()

    # Using full train_df (no sampling)
    train_df = df.filter(F.col(FOLD_COL) != k)
    test_df = df.filter(F.col(FOLD_COL) == k)

    pred = bagging_train_predict(train_df, test_df, FEATURES_COL, LABEL_COL,
                                 num_trees=NUM_TREES, max_depth=MAX_DEPTH)

    m = compute_metrics(pred, LABEL_COL, "prediction")
    m["fold"] = k
    results.append(m)
    print(f"[{i}/{len(all_folds)}] Fold {k} Finished | Acc: {m['accuracy']:.4f} | Time: {time.time() - fold_start:.1f}s")

# 6. FINAL RESULTS
# ------------------------------------------------------
results_df = pd.DataFrame(results)
print(f"\nALL FOLDS FINISHED! Total Time: {(time.time() - start_time)/60:.2f} minutes")
print("\nFinal Average Metrics:")
print(results_df[["accuracy", "precision", "recall", "f1"]].mean())

metrics_dir = os.path.join(BASE_DIR, "outputs", "metrics")
os.makedirs(metrics_dir, exist_ok=True)

out_path = os.path.join(metrics_dir, "bagging_results_FINAL.csv")
results_df.to_csv(out_path, index=False)
print("Final results saved to", out_path)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 21:32:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/12 21:32:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/12 21:32:38 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/12 21:32:38 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark Session Created Successfully!


Full Dataset Loaded. Total Rows: 88092
Running FULL Bagging Ensemble on all 10 folds...
Using 100% of Training Data per fold.
Forest: 50 trees, Depth 10


[1/10] Fold 0 Finished | Acc: 0.8148 | Time: 515.3s


[2/10] Fold 1 Finished | Acc: 0.8043 | Time: 635.0s


...

[6/10] Fold 5 Finished | Acc: 0.8043 | Time: 533.8s


[7/10] Fold 6 Finished | Acc: 0.8069 | Time: 546.3s


[8/10] Fold 7 Finished | Acc: 0.8094 | Time: 525.6s


[9/10] Fold 8 Finished | Acc: 0.8015 | Time: 521.2s


[10/10] Fold 9 Finished | Acc: 0.7977 | Time: 529.3s

ALL FOLDS FINISHED! Total Time: 117.21 minutes

Final Average Metrics:
accuracy     0.805476
precision    0.807888
recall       0.805433
f1           0.806160
dtype: float64
Final results saved to /app/outputs/metrics/bagging_results_FINAL.csv
